# Relation Extraction And Entity Linking

This notebook covers relation extraction beyond entity candidates: OpenIE-style triples, dependency patterns, GLiNER spans plus Ollama relation typing, REBEL-style transformer adapters, and entity linking/resolution against external or semantic signals.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "src").exists():
            ROOT = parent
            break
sys.path.insert(0, str(ROOT / "src"))


## OpenIE and dependency-pattern relations

OpenIE gives open-domain subject-relation-object candidates. Dependency-pattern extraction is narrower but more predictable for enterprise text.


In [ ]:
from __future__ import annotations

import json
import urllib.parse
from typing import Any

from kg_construction.domain.evidence import span_evidence
from kg_construction.domain.models import ConstructionBatch, EntityCandidate, SourceRecord
from kg_construction.infrastructure.extractors import (
    DependencyPatternRelationExtractor,
    GlinerOllamaRelationTypingExtractor,
    OpenIERelationExtractor,
    RebelRelationExtractor,
    RelationTypingConfig,
    TransformerRelationExtractor,
)
from kg_construction.infrastructure.llm.ollama import OllamaConfig, OllamaProvider
from kg_construction.infrastructure.resolution import (
    AliasTableResolver,
    CrossEncoderResolver,
    CrossEncoderResolverConfig,
    DBpediaLinker,
    DedupeActiveLearningResolver,
    EmbeddingResolver,
    EmbeddingResolverConfig,
    NormalizedStringResolver,
    OpenRefineReconciler,
    OpenRefineReconcilerConfig,
    RapidFuzzResolver,
    RapidFuzzResolverConfig,
    SplinkProbabilisticResolver,
    WikidataLinker,
)

openie_source = SourceRecord.from_text(
    "Ada Lovelace works at Analytical Engine Lab. Grace Hopper reports to Ada Lovelace.",
    source_name="memory://relations",
)
openie_batch = OpenIERelationExtractor().extract(openie_source)
dependency_batch = DependencyPatternRelationExtractor().extract(openie_source)
{
    "openie": [(r.properties["source_name"], r.relation_type, r.properties["target_name"]) for r in openie_batch.relations],
    "dependency": [(r.properties["source_name"], r.relation_type, r.properties["target_name"]) for r in dependency_batch.relations],
}


## GLiNER entities + Ollama relation typing

The hybrid path first extracts spans with GLiNER labels, then asks an LLM to type only evidence-backed relations between those spans. The fake GLiNER model and fake Ollama transport keep the notebook offline.


In [ ]:
class FakeGlinerModel:
    def predict_entities(self, text: str, labels: list[str], *, threshold: float = 0.5, flat_ner: bool = True):
        _ = threshold, flat_ner
        spans = {"Incident": "INCIDENT-1042", "System": "Recommendation API", "Date": "2026-07-09"}
        entities = []
        for label, value in spans.items():
            if label in labels and value in text:
                start = text.find(value)
                entities.append({"text": value, "label": label, "start": start, "end": start + len(value), "score": 0.92})
        return entities


def fake_ollama_relation_transport(body: dict[str, object]) -> dict[str, object]:
    return {
        "message": {
            "content": json.dumps(
                {
                    "relations": [
                        {
                            "source": "INCIDENT-1042",
                            "type": "AFFECTED",
                            "target": "Recommendation API",
                            "evidence_text": "INCIDENT-1042 affected Recommendation API",
                            "confidence": 0.94,
                        }
                    ]
                }
            )
        }
    }

hybrid_source = SourceRecord.from_text(
    "INCIDENT-1042 affected Recommendation API on 2026-07-09.",
    source_name="memory://hybrid-relations",
)
hybrid_batch = GlinerOllamaRelationTypingExtractor(
    OllamaProvider(OllamaConfig(model="offline-fake"), transport=fake_ollama_relation_transport),
    config=RelationTypingConfig(
        entity_labels=["Incident", "System", "Date"],
        allowed_relationship_types=["AFFECTED", "RELATED_TO"],
    ),
    gliner_model=FakeGlinerModel(),
).extract(hybrid_source)
{
    "entities": [(e.name, e.entity_type, e.evidence[0].location) for e in hybrid_batch.entities],
    "relations": [(r.properties["source_name"], r.relation_type, r.properties["target_name"]) for r in hybrid_batch.relations],
}


## REBEL / transformer relation adapters

The REBEL adapter is an integration boundary for transformer relation extraction. Injecting a pipeline makes the adapter testable without downloading a model.


In [ ]:
class FakeRebelPipeline:
    def __call__(self, text: str, **kwargs: object):
        _ = text, kwargs
        return [{"generated_text": "<triplet> Ada Lovelace <subj> Analytical Engine <obj> created"}]

rebel_source = SourceRecord.from_text("Ada Lovelace created Analytical Engine.", source_name="memory://rebel")
rebel_batch = RebelRelationExtractor(pipeline=FakeRebelPipeline()).extract(rebel_source)
transformer_batch = TransformerRelationExtractor(pipeline=FakeRebelPipeline()).extract(rebel_source)
{
    "rebel": [(r.properties["source_name"], r.relation_type, r.properties["target_name"]) for r in rebel_batch.relations],
    "transformer_adapter": transformer_batch.metadata["extractor"],
}


## Entity linking and semantic resolution

This block demonstrates external linkers with fake HTTP transports plus embedding/cross-encoder resolvers with tiny in-memory models. Splink and Dedupe are shown as adapter boundaries for project-specific linkage work.


In [ ]:
def batch_with_entities(*names: str, entity_type: str = "System") -> ConstructionBatch:
    text = " | ".join(names)
    source = SourceRecord.from_text(text, source_name="memory://entity-linking")
    entities = []
    cursor = 0
    for name in names:
        start = text.find(name, cursor)
        cursor = start + len(name)
        entities.append(
            EntityCandidate.build(
                entity_type=entity_type,
                name=name,
                evidence=[span_evidence(source, start=start, end=start + len(name), text=name)],
                extractor_name="notebook",
                technique="fixture",
            )
        )
    return ConstructionBatch(source_records=[source], entities=entities)

class FakeEmbeddingModel:
    def encode(self, texts: list[str]) -> list[list[float]]:
        return [[1.0, 0.0] if "recommend" in text.casefold() or "recommender" in text.casefold() else [0.0, 1.0] for text in texts]

class FakeCrossEncoderModel:
    def predict(self, pairs: list[tuple[str, str]]) -> list[float]:
        return [0.96 if "recommendation" in left.casefold() and "recommender" in right.casefold() else 0.2 for left, right in pairs]


def fake_wikidata(url: str, **kwargs: Any) -> dict[str, object]:
    return {"search": [{"id": "Q123", "label": "Recommendation API", "description": "software service", "concepturi": "https://www.wikidata.org/entity/Q123"}]}


def fake_dbpedia(url: str, **kwargs: Any) -> dict[str, object]:
    return {"docs": [{"resource": ["http://dbpedia.org/resource/Feature_store"], "label": ["Feature Store"], "comment": ["Feature stores manage ML features."]}]}


def fake_openrefine(url: str, **kwargs: Any) -> dict[str, object]:
    body = kwargs["body"].decode("utf-8")
    queries = json.loads(urllib.parse.parse_qs(body)["queries"][0])
    return {key: {"result": [{"id": "wd:Q37156", "name": "International Business Machines", "score": 98, "match": True}]} for key in queries}

link_batch = batch_with_entities("Recommendation API")
wikidata_batch = WikidataLinker(transport=fake_wikidata).resolve(link_batch)
dbpedia_batch = DBpediaLinker(transport=fake_dbpedia).resolve(batch_with_entities("Feature Store"))
openrefine_batch = OpenRefineReconciler(OpenRefineReconcilerConfig(), transport=fake_openrefine).resolve(batch_with_entities("IBM", "International Business Machines", entity_type="Organization"))
semantic_batch = batch_with_entities("Recommendation API", "Recommender API", "Feature Store")
embedding_batch = EmbeddingResolver(EmbeddingResolverConfig(threshold=0.95), model=FakeEmbeddingModel()).resolve(semantic_batch)
cross_encoder_batch = CrossEncoderResolver(CrossEncoderResolverConfig(threshold=0.9), model=FakeCrossEncoderModel()).resolve(semantic_batch)

adapter_notes = []
for resolver in [SplinkProbabilisticResolver(), DedupeActiveLearningResolver()]:
    try:
        resolver.resolve(semantic_batch)
    except Exception as exc:
        adapter_notes.append(f"{resolver.name}: {type(exc).__name__}")

{
    "wikidata_id": wikidata_batch.entities[0].properties.get("wikidata_id"),
    "dbpedia_id": dbpedia_batch.entities[0].properties.get("dbpedia_id"),
    "openrefine_canonical_ids": sorted({e.properties.get("canonical_entity_id") for e in openrefine_batch.entities}),
    "embedding_canonical_ids": sorted({e.properties.get("canonical_entity_id") for e in embedding_batch.entities if "API" in e.name}),
    "cross_encoder_audit_method": cross_encoder_batch.metadata["entity_resolution_audit"][0]["method"],
    "adapter_boundaries": adapter_notes,
}
